# Export Your Data for Power BI

"I exported clean fact tables for Power BI from the enhanced dataset. The export uses enhanced PD (5.76%) and LGD (14.18%) from Notebook 01, and actual scenario results from Notebook 04: Baseline ($1,863), Adverse ($4,024), Severely Adverse ($8,383), and Tail Risk ($12,938). The export includes 11 files: loan portfolio, segment analysis, scenario analysis, capital adequacy, IFRS 9 stages, fairness testing, date dimension, segment dimension, and summary metrics. All exports include a complete audit trail for reproducibility."

**In Notebook 06, I exported data for Power BI dashboards. This is the bridge between my Python analysis and business users.**

1. **I exported 9 CSV files following a star schema design — fact tables (measurements) and dimension tables (attributes). The fact tables include loan-level data, segment analysis, scenario ECL, capital adequacy, IFRS 9 stages, and fairness testing. The dimension tables include date and segment dimensions.**

2. **I also exported a summary metrics table with 17 KPIs — portfolio health, IFRS 9 parameters, capital adequacy, and fairness testing. This enables executive dashboards.**

3. **I documented the limitations — the exports are static, the data is a 100,000-row sample, and the ECL is estimated. Power BI users need to understand the context.**

4. **The output is a complete set of CSV files that can be imported into Power BI Desktop for interactive dashboards.**

**This shows that I understand that data science work needs to be accessible to business users — not just data scientists.**

In [1]:
"""
===============================================================================
FANNIE MAE IFRS 9 COMPLIANCE FRAMEWORK
NOTEBOOK 06: POWER BI DATA EXPORT
===============================================================================

AUTHOR: [Your Name]
DATE: [Current Date]
VERSION: 2.0 (Revised)

PURPOSE:
--------
Export clean fact tables for Power BI dashboard visualization. This notebook
transforms the final dataset and calibration results into CSV files that can
be directly imported into Power BI Desktop for interactive reporting and
dashboard creation.

INPUTS:
-------
- data/fannie_mae_final_clean.csv : Cleaned dataset from Notebook 01
- data/assumptions_register.csv   : Central assumptions register
- outputs/pd_lgd_ead_calibration_summary.txt : Calibration results
- outputs/stress_testing_summary.txt : Stress testing results

OUTPUTS:
--------
- powerbi_exports/fact_loan_portfolio.csv      : Loan-level data
- powerbi_exports/fact_segment_analysis.csv    : Segment risk analysis
- powerbi_exports/fact_scenario_analysis.csv   : Scenario ECL results
- powerbi_exports/fact_capital_adequacy.csv    : Capital adequacy metrics
- powerbi_exports/fact_ifrs9_stages.csv        : Stage distribution
- powerbi_exports/fact_fairness_testing.csv    : Fairness test results
- powerbi_exports/dim_date.csv                 : Date dimension
- powerbi_exports/dim_segment.csv              : Segment dimension
- powerbi_exports/summary_metrics.csv          : Key summary metrics
- logs/powerbi_export_audit_trail_{RUN_ID}.log : Audit trail

REGULATORY CONTEXT:
-------------------
- OSFI E-23 s.4.2: Documentation for all outputs
- OSFI E-23 s.4.3: Data quality for reporting
- IFRS 9: ECL disclosure and reporting
- SR 11-7: Model validation outcomes

DATASET CAVEAT (REFERENCED):
----------------------------
This notebook uses the dataset described in Notebook 01. As stated there,
this project uses public Fannie Mae data as a PROXY for proprietary bank data.
All conclusions are illustrative and demonstrate methodology, not actual
portfolio performance.

For the full caveat, see Notebook 01 — Section 2: CECL vs IFRS 9 Context.

US/CANADA APPLICABILITY GAP (REFERENCED):
-----------------------------------------
For important differences between US and Canadian mortgage markets, and the
implications for this analysis, see Notebook 01 — Section 2.1:
US/Canada Applicability Gap.

===============================================================================
"""

'\n===============================================================================\nFANNIE MAE IFRS 9 COMPLIANCE FRAMEWORK\nNOTEBOOK 06: POWER BI DATA EXPORT\n===============================================================================\n\nAUTHOR: [Your Name]\nDATE: [Current Date]\nVERSION: 2.0 (Revised)\n\nPURPOSE:\n--------\nExport clean fact tables for Power BI dashboard visualization. This notebook\ntransforms the final dataset and calibration results into CSV files that can\nbe directly imported into Power BI Desktop for interactive reporting and\ndashboard creation.\n\nINPUTS:\n-------\n- data/fannie_mae_final_clean.csv : Cleaned dataset from Notebook 01\n- data/assumptions_register.csv   : Central assumptions register\n- outputs/pd_lgd_ead_calibration_summary.txt : Calibration results\n- outputs/stress_testing_summary.txt : Stress testing results\n\nOUTPUTS:\n--------\n- powerbi_exports/fact_loan_portfolio.csv      : Loan-level data\n- powerbi_exports/fact_segment_analysis.csv

#### What It Means
This header tells anyone opening the notebook:

- What this notebook does (exports data for Power BI dashboards)
- What it needs (clean dataset, assumptions register, calibration results, stress testing results)
- What it produces (9 CSV files for Power BI)
- What regulations apply (OSFI E-23, IFRS 9, SR 11-7)
- What the limitations are (proxy data)

#### Why This Decision Was Made:
- **Why a Power BI export notebook?** Data scientists don't work in isolation — business users need to explore data in tools like Power BI. This notebook makes your work accessible.
- **Why 9 CSV files?** This follows star schema design — fact tables (measurements) and dimension tables (attributes) that Power BI can relate.
- **Why reference the dataset caveat?** Users of the Power BI dashboard need to know this is proxy data, not actual bank data.
- **Why list regulatory context?** Even the export notebook shows regulatory awareness — documentation is required for all outputs.

**I understand that data science work needs to be accessible to business users. I export data to Power BI for interactive dashboards.**

In [2]:
import pandas as pd
import numpy as np
import os
import logging
from datetime import datetime
import sys

### Setup Audit Trail

In [3]:
def setup_audit_logger(run_id=None):
    """Set up comprehensive audit trail logging."""
    if run_id is None:
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    os.makedirs('logs', exist_ok=True)
    
    logger = logging.getLogger(f"powerbi_export_{run_id}")
    logger.setLevel(logging.INFO)
    
    if logger.hasHandlers():
        logger.handlers.clear()
    
    handler = logging.FileHandler(f"logs/powerbi_export_audit_trail_{run_id}.log", encoding='utf-8')
    formatter = logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s', 
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    
    return logger, run_id

audit_logger, RUN_ID = setup_audit_logger()
audit_logger.info("="*60)
audit_logger.info("POWER BI DATA EXPORT")
audit_logger.info(f"RUN ID: {RUN_ID}")
audit_logger.info("="*60)

print("\n✅ Audit trail initialized: logs/powerbi_export_audit_trail_{RUN_ID}.log".format(RUN_ID=RUN_ID))

# Create export directory
os.makedirs('powerbi_exports', exist_ok=True)
print("\n✅ Export directory created: powerbi_exports/")
audit_logger.info("Export directory created: powerbi_exports/")

2026-09-07 18:11:07 | INFO | ============================================================
2026-09-07 18:11:07 | INFO | POWER BI DATA EXPORT
2026-09-07 18:11:07 | INFO | RUN ID: 20260907_181107
2026-09-07 18:11:07 | INFO | ============================================================

✅ Audit trail initialized: logs/powerbi_export_audit_trail_20260907_181107.log

✅ Export directory created: powerbi_exports/
2026-09-07 18:11:07 | INFO | Export directory created: powerbi_exports/


#### What It Means

Same pattern as previous notebooks — creates a logging system with a unique run ID. It also creates the powerbi_exports/ directory where the CSV files will be saved.

#### Why This Decision Was Made
- **Why a separate log file?** powerbi_export_audit_trail_*.log keeps export logs separate from analysis logs.
- **Why create the directory explicitly?** If the directory doesn't exist, the export will fail. Creating it explicitly prevents errors.
- **Why log the directory creation?** The audit trail should record what was created and when.

**I maintain a consistent audit trail — even for data exports. Every output is traceable.**

### SECTION 1: LOAD DATA & ASSUMPTIONS

In [4]:
print("\n" + "="*60)
print("SECTION 1: LOADING DATA AND ASSUMPTIONS")
print("="*60)

audit_logger.info("SECTION 1: LOADING DATA AND ASSUMPTIONS")

# Load the assumptions register for reference
try:
    assumptions_df = pd.read_csv('data/assumptions_register.csv')
    print(f"✅ Loaded assumptions register: {len(assumptions_df)} entries")
    audit_logger.info(f"Loaded assumptions register: {len(assumptions_df)} entries")
except FileNotFoundError:
    print("⚠️  Assumptions register not found. Please run Notebook 01 first.")
    audit_logger.warning("Assumptions register not found")
    assumptions_df = pd.DataFrame()

# Load enhanced final dataset
try:
    df = pd.read_csv('data/fannie_mae_final_clean.csv')
    print(f"✅ Loaded enhanced data: {df.shape}")
    print(f"   Default Rate: {df['default'].mean():.2%}")
    print(f"   Columns: {len(df.columns)}")
    audit_logger.info(f"Data loaded: {df.shape}")
    audit_logger.info(f"Default Rate: {df['default'].mean():.2%}")
except FileNotFoundError:
    print("⚠️  Final clean data not found. Trying cleaned data...")
    audit_logger.warning("Final clean data not found, trying cleaned data")
    try:
        df = pd.read_csv('data/fannie_mae_cleaned.csv')
        print(f"✅ Loaded cleaned data: {df.shape}")
        audit_logger.info(f"Loaded cleaned data: {df.shape}")
    except FileNotFoundError:
        print("❌ Data file not found. Please run Notebook 01 first.")
        audit_logger.error("Data file not found")
        raise



SECTION 1: LOADING DATA AND ASSUMPTIONS
2026-09-07 18:11:07 | INFO | SECTION 1: LOADING DATA AND ASSUMPTIONS
✅ Loaded assumptions register: 10 entries
2026-09-07 18:11:07 | INFO | Loaded assumptions register: 10 entries
✅ Loaded enhanced data: (100000, 179)
   Default Rate: 0.94%
   Columns: 179
2026-09-07 18:11:07 | INFO | Data loaded: (100000, 179)
2026-09-07 18:11:07 | INFO | Default Rate: 0.94%


#### What It Means

This loads:

- The assumptions register from Notebook 01 (for documentation)
- The clean dataset — with PD-LGD estimates from Notebook 03

#### Why This Decision Was Made:
- **Why load the assumptions register?** If someone asks "where did these numbers come from?", you can point to the assumptions register.
- **Why load from fannie_mae_final_clean.csv?** This is the final cleaned dataset — all features are ready.
- **Why the fallback to cleaned data?** If the final dataset is missing, try the cleaned dataset. This is robustness.
- **Why raise an error if no data is found?** This notebook can't proceed without data. It's better to fail early and clearly.

**I load data from the final output of previous notebooks. I handle errors gracefully and fail clearly when data is missing.**

### SECTION 2: SET PARAMETERS

In [5]:
print("\n" + "="*60)
print("SECTION 2: SETTING PARAMETERS")
print("="*60)

audit_logger.info("SECTION 2: SETTING PARAMETERS")

"""
DECISION: Use enhanced PD estimate if available
RATIONALE: Notebook 01 created PD estimates that are more sophisticated
           than the simple default rate. These should be used for Power BI
           reporting.
REGULATORY REFERENCE: IFRS 9 s.5.5.3
"""
if 'pd_estimate' in df.columns:
    base_pd = df['pd_estimate'].mean()
    print(f"✅ Using enhanced PD estimate: {base_pd:.2%}")
    audit_logger.info(f"Using enhanced PD: {base_pd:.2%}")
else:
    base_pd = df['default'].mean()
    print(f"⚠️  Using base PD estimate: {base_pd:.2%}")
    audit_logger.info(f"Using base PD: {base_pd:.2%}")

"""
DECISION: Use enhanced LGD estimate if available
RATIONALE: Notebook 01 created LGD estimates with collateral segmentation.
           These should be used for Power BI reporting.
REGULATORY REFERENCE: IFRS 9 s.5.5.3
"""
if 'lgd_final' in df.columns:
    base_lgd = df['lgd_final'].mean()
    print(f"✅ Using enhanced LGD estimate: {base_lgd:.2%}")
    audit_logger.info(f"Using enhanced LGD: {base_lgd:.2%}")
else:
    base_lgd = 0.45
    print(f"⚠️  Using base LGD estimate: {base_lgd:.2%}")
    audit_logger.info(f"Using base LGD: {base_lgd:.2%}")

base_ead = df['ORG_UPB'].mean() if 'ORG_UPB' in df.columns else 200000
print(f"✅ EAD: ${base_ead:,.0f}")

print(f"\nBase Parameters Summary:")
print(f"  PD: {base_pd:.2%}")
print(f"  LGD: {base_lgd:.2%}")
print(f"  EAD: ${base_ead:,.0f}")
audit_logger.info(f"Base Parameters: PD={base_pd:.2%}, LGD={base_lgd:.2%}, EAD=${base_ead:,.0f}")


SECTION 2: SETTING PARAMETERS
2026-09-07 18:11:07 | INFO | SECTION 2: SETTING PARAMETERS
✅ Using enhanced PD estimate: 5.76%
2026-09-07 18:11:08 | INFO | Using enhanced PD: 5.76%
✅ Using enhanced LGD estimate: 14.18%
2026-09-07 18:11:08 | INFO | Using enhanced LGD: 14.18%
✅ EAD: $236,040

Base Parameters Summary:
  PD: 5.76%
  LGD: 14.18%
  EAD: $236,040
2026-09-07 18:11:08 | INFO | Base Parameters: PD=5.76%, LGD=14.18%, EAD=$236,040


#### What It Means

This sets the base parameters for the Power BI exports:

- PD: 5.76% (using the enhanced PD estimate from Notebook 01)
- LGD: 14.18% (using the enhanced LGD estimate from Notebook 01)
- EAD: $236,040 (average loan amount)

#### Why This Decision Was Made:
- **Why use enhanced PD and LGD?** The simple default rate (0.94%) is less sophisticated than the calibrated PD (5.76%). Power BI users should see the best available numbers.
- **Why the fallback to simple values?** If the enhanced estimates aren't available, fall back to simple values. The notebook should still run.
- **Why document the decision?** A Power BI user might wonder "where did 5.76% come from?" The decision log explains it.
- **Why log the parameters?** The audit trail records what parameters were used for the export.

**I use the best available data for Power BI — the enhanced PD and LGD estimates, not the simple default rate. I document my decisions.**

### SECTION 3: FACT TABLE - LOAN PORTFOLIO

### SECTION 3.1: FACT TABLE — LOAN PORTFOLIO

In [6]:
print("\n" + "="*60)
print("SECTION 3: EXPORTING FACT TABLES")
print("="*60)

audit_logger.info("SECTION 3: EXPORTING FACT TABLES")

"""
DECISION: Export loan-level data for Power BI
RATIONALE: Allows drill-down analysis in Power BI dashboards
REGULATORY REFERENCE: OSFI E-23 s.4.2
"""

print("\n" + "-"*60)
print("3.1 FACT TABLE: LOAN PORTFOLIO")
print("-"*60)
audit_logger.info("Exporting fact_loan_portfolio")

# Prepare loan-level data for Power BI
loan_fact = df[['default', 'ORG_UPB', 'ORG_LTV', 'DTI', 'FICO_BOR', 'AGE', 
                'PROP_TYPE', 'OCCU_STAT']].copy()

# Add derived fields for easier analysis
loan_fact['default_flag'] = loan_fact['default'].map({0: 'Performing', 1: 'Default'})
loan_fact['fico_band'] = pd.cut(loan_fact['FICO_BOR'], 
                                 bins=[620, 680, 720, 760, 832],
                                 labels=['Low', 'Medium', 'High', 'Very High'])
loan_fact['ltv_band'] = pd.cut(loan_fact['ORG_LTV'],
                                bins=[0, 60, 80, 100],
                                labels=['Low', 'Medium', 'High'])
loan_fact['dti_category'] = loan_fact['DTI'].apply(lambda x: '>43%' if x > 43 else '<=43%')

# Add age bands
def age_band(age):
    if age < 12: return '0-12m'
    elif age < 24: return '12-24m'
    elif age < 36: return '24-36m'
    elif age < 60: return '36-60m'
    elif age < 120: return '60-120m'
    else: return '120m+'

loan_fact['age_band'] = loan_fact['AGE'].apply(age_band)

# Add enhanced features if available
if 'pd_estimate' in df.columns:
    loan_fact['pd_estimate'] = df['pd_estimate']
if 'lgd_final' in df.columns:
    loan_fact['lgd_final'] = df['lgd_final']
if 'delinquency_state' in df.columns:
    loan_fact['delinquency_state'] = df['delinquency_state']
    loan_fact['delinquency_label'] = df['delinquency_label']
if 'ifrs9_stage' in df.columns:
    loan_fact['ifrs9_stage'] = df['ifrs9_stage']
if 'sicr_trigger' in df.columns:
    loan_fact['sicr_trigger'] = df['sicr_trigger']
if 'pd_estimate' in df.columns and 'lgd_final' in df.columns:
    loan_fact['estimated_ecl'] = df['pd_estimate'] * df['lgd_final'] * loan_fact['ORG_UPB']

loan_fact.to_csv('powerbi_exports/fact_loan_portfolio.csv', index=False)
print(f"✅ Exported fact_loan_portfolio.csv ({len(loan_fact):,} rows)")
audit_logger.info(f"Exported fact_loan_portfolio.csv ({len(loan_fact):,} rows)")


SECTION 3: EXPORTING FACT TABLES
2026-09-07 18:11:08 | INFO | SECTION 3: EXPORTING FACT TABLES

------------------------------------------------------------
3.1 FACT TABLE: LOAN PORTFOLIO
------------------------------------------------------------
2026-09-07 18:11:08 | INFO | Exporting fact_loan_portfolio
✅ Exported fact_loan_portfolio.csv (100,000 rows)
2026-09-07 18:11:08 | INFO | Exported fact_loan_portfolio.csv (100,000 rows)


#### What It Means

This exports loan-level data for Power BI:

- Core variables: default status, loan amount, LTV, DTI, FICO, age, property type, occupancy
- Derived fields: FICO band, LTV band, DTI category, age band
- Enhanced features: PD estimate, LGD estimate, delinquency state, IFRS 9 stage, SICR trigger
- Estimated ECL: PD × LGD × loan amount

#### Why This Decision Was Made:
- **Why export loan-level data?** Power BI users can drill down to individual loans — filter by FICO band, property type, or stage.
- **Why add derived fields?** Derived fields (FICO band, LTV band, age band) make it easier to group and filter in Power BI.
- **Why include IFRS 9 stage and SICR trigger?** Power BI users can see the stage distribution and why loans are in Stage 2/3.
- **Why calculate estimated ECL?** This allows Power BI users to see ECL at the loan level — which loans are driving ECL.
- **Why save without index?** The DataFrame index is meaningless in Power BI — saving without it avoids confusion.

**I export loan-level data with derived fields and enhanced features. Power BI users can drill down to individual loans and understand why they're in each stage.**

### SECTION 3.2: FACT TABLE — SEGMENT ANALYSIS

In [7]:
print("\n" + "-"*60)
print("3.2 FACT TABLE: SEGMENT ANALYSIS")
print("-"*60)
audit_logger.info("Exporting fact_segment_analysis")

# Calculate segment-level default rates
segment_data = []

# FICO bands
for band in ['Low', 'Medium', 'High', 'Very High']:
    subset = loan_fact[loan_fact['fico_band'] == band]
    if len(subset) > 0:
        segment_data.append({
            'Segment_Type': 'FICO',
            'Category': band,
            'Default_Rate': subset['default'].mean(),
            'Count': len(subset),
            'Avg_PD': subset['pd_estimate'].mean() if 'pd_estimate' in subset.columns else subset['default'].mean(),
            'Avg_LGD': subset['lgd_final'].mean() if 'lgd_final' in subset.columns else base_lgd,
            'Avg_ECL': subset['estimated_ecl'].mean() if 'estimated_ecl' in subset.columns else 0,
            'Risk_Level': 'High' if subset['default'].mean() > loan_fact['default'].mean() * 1.5 else 'Normal'
        })

# LTV bands
for band in ['Low', 'Medium', 'High']:
    subset = loan_fact[loan_fact['ltv_band'] == band]
    if len(subset) > 0:
        segment_data.append({
            'Segment_Type': 'LTV',
            'Category': band,
            'Default_Rate': subset['default'].mean(),
            'Count': len(subset),
            'Avg_PD': subset['pd_estimate'].mean() if 'pd_estimate' in subset.columns else subset['default'].mean(),
            'Avg_LGD': subset['lgd_final'].mean() if 'lgd_final' in subset.columns else base_lgd,
            'Avg_ECL': subset['estimated_ecl'].mean() if 'estimated_ecl' in subset.columns else 0,
            'Risk_Level': 'High' if subset['default'].mean() > loan_fact['default'].mean() * 1.5 else 'Normal'
        })

# Property types
for prop in loan_fact['PROP_TYPE'].unique():
    subset = loan_fact[loan_fact['PROP_TYPE'] == prop]
    if len(subset) > 0:
        segment_data.append({
            'Segment_Type': 'Property Type',
            'Category': prop,
            'Default_Rate': subset['default'].mean(),
            'Count': len(subset),
            'Avg_PD': subset['pd_estimate'].mean() if 'pd_estimate' in subset.columns else subset['default'].mean(),
            'Avg_LGD': subset['lgd_final'].mean() if 'lgd_final' in subset.columns else base_lgd,
            'Avg_ECL': subset['estimated_ecl'].mean() if 'estimated_ecl' in subset.columns else 0,
            'Risk_Level': 'High' if subset['default'].mean() > loan_fact['default'].mean() * 1.5 else 'Normal'
        })

# Age bands
age_bands = ['0-12m', '12-24m', '24-36m', '36-60m', '60-120m', '120m+']
for band in age_bands:
    subset = loan_fact[loan_fact['age_band'] == band]
    if len(subset) > 0:
        segment_data.append({
            'Segment_Type': 'Loan Age',
            'Category': band,
            'Default_Rate': subset['default'].mean(),
            'Count': len(subset),
            'Avg_PD': subset['pd_estimate'].mean() if 'pd_estimate' in subset.columns else subset['default'].mean(),
            'Avg_LGD': subset['lgd_final'].mean() if 'lgd_final' in subset.columns else base_lgd,
            'Avg_ECL': subset['estimated_ecl'].mean() if 'estimated_ecl' in subset.columns else 0,
            'Risk_Level': 'High' if subset['default'].mean() > loan_fact['default'].mean() * 1.5 else 'Normal'
        })

df_segments = pd.DataFrame(segment_data)
df_segments.to_csv('powerbi_exports/fact_segment_analysis.csv', index=False)
print(f"✅ Exported fact_segment_analysis.csv ({len(df_segments)} rows)")
audit_logger.info(f"Exported fact_segment_analysis.csv ({len(df_segments)} rows)")


------------------------------------------------------------
3.2 FACT TABLE: SEGMENT ANALYSIS
------------------------------------------------------------
2026-09-07 18:11:08 | INFO | Exporting fact_segment_analysis
✅ Exported fact_segment_analysis.csv (15 rows)
2026-09-07 18:11:09 | INFO | Exported fact_segment_analysis.csv (15 rows)


#### What It Means:

This exports segment-level analysis for Power BI:

- FICO bands: Low, Medium, High, Very High
- LTV bands: Low, Medium, High
- Property types: SF, PU, CO, MH, CP
- Loan age bands: 0-12m, 12-24m, etc.

For each segment, it includes:

- Default rate
- Count
- Average PD, LGD, ECL
- Risk level (High or Normal)

#### Why This Decision Was Made:
- **Why export segment-level data?** Power BI users can compare risk across segments — which FICO band has the highest default rate?
- **Why include PD, LGD, and ECL?** This allows Power BI users to see why a segment is risky — high PD, high LGD, or both.
- **Why the risk level classification?** Risk level (High/Normal) makes it easy to filter for high-risk segments.
- **Why the if len(subset) > 0?** If a segment has no loans, skip it. This prevents errors.
What a Hiring Manager Hears:

**I export segment-level analysis so Power BI users can compare risk across FICO bands, LTV bands, property types, and loan ages.**

### SECTION 3.3: FACT TABLE — SCENARIO ANALYSIS

In [8]:
print("\n" + "-"*60)
print("3.3 FACT TABLE: SCENARIO ANALYSIS")
print("-"*60)
audit_logger.info("Exporting fact_scenario_analysis")

"""
DECISION: Use actual scenario results from Notebook 04
RATIONALE: Ensures consistency between notebooks and Power BI
REGULATORY REFERENCE: IFRS 9 s.5.5.17
"""

scenario_data = {
    'Scenario': ['Baseline', 'Adverse', 'Severely Adverse', 'Tail Risk'],
    'PD_Multiplier': [1.0, 1.8, 3.0, 4.5],
    'LGD_Multiplier': [1.0, 1.2, 1.5, 2.0],
    'ECL': [1863, 4024, 8383, 12938],
    'Weight': [0.45, 0.30, 0.15, 0.10],
    'Description': [
        'Current economic outlook',
        'Moderate recession',
        'Severe recession',
        'Extreme stress'
    ]
}
df_scenarios = pd.DataFrame(scenario_data)

# Calculate weighted ECL
df_scenarios['Weighted_ECL'] = df_scenarios['ECL'] * df_scenarios['Weight']
df_scenarios['ECL_Multiplier'] = df_scenarios['ECL'] / df_scenarios['ECL'].iloc[0]

df_scenarios.to_csv('powerbi_exports/fact_scenario_analysis.csv', index=False)
print(f"✅ Exported fact_scenario_analysis.csv ({len(df_scenarios)} rows)")
audit_logger.info(f"Exported fact_scenario_analysis.csv ({len(df_scenarios)} rows)")



------------------------------------------------------------
3.3 FACT TABLE: SCENARIO ANALYSIS
------------------------------------------------------------
2026-09-07 18:11:09 | INFO | Exporting fact_scenario_analysis
✅ Exported fact_scenario_analysis.csv (4 rows)
2026-09-07 18:11:09 | INFO | Exported fact_scenario_analysis.csv (4 rows)


#### What It Means:

This exports scenario analysis results:

- 4 scenarios: Baseline, Adverse, Severely Adverse, Tail Risk
- PD and LGD multipliers: 1.0×, 1.8×, 3.0×, 4.5×
- ECL and weighted ECL: $1,863 to $12,938
- Weights: 45%, 30%, 15%, 10%

#### Why This Decision Was Made:
- **Why use actual scenario results from Notebook 04?** Consistency — Power BI should show the same results as the analysis notebooks.
- **Why include PD and LGD multipliers?** Power BI users can see how much stress is applied in each scenario.
- **Why include weighted ECL?** Weighted ECL is the probability-weighted result — it's the most meaningful number.
- **Why include description?** Descriptions make the scenarios understandable to non-technical users.

**I export scenario analysis results so Power BI users can see ECL under different economic conditions.**

### SECTION 3.4: FACT TABLE — CAPITAL ADEQUACY

In [9]:
print("\n" + "-"*60)
print("3.4 FACT TABLE: CAPITAL ADEQUACY")
print("-"*60)
audit_logger.info("Exporting fact_capital_adequacy")

"""
DECISION: Show capital adequacy under each scenario
RATIONALE: Enables capital planning visualization in Power BI
REGULATORY REFERENCE: Basel III
"""

# Capital calculations
total_assets = 1_000_000_000
cet1_ratio = 0.045
cet1_capital = total_assets * cet1_ratio

capital_data = []
for scenario in df_scenarios['Scenario']:
    ecl = df_scenarios[df_scenarios['Scenario'] == scenario]['ECL'].values[0]
    capital_data.append({
        'Scenario': scenario,
        'ECL': ecl,
        'CET1_Capital': cet1_capital,
        'Capital_Impact_Ratio': ecl / cet1_capital,
        'Capital_Buffer': cet1_capital - ecl,
        'Adequacy': 'Adequate' if ecl < cet1_capital else 'Inadequate',
        'CET1_Ratio': cet1_ratio
    })

# Add weighted scenario
weighted_ecl = df_scenarios['Weighted_ECL'].sum()
capital_data.append({
    'Scenario': 'Probability-Weighted',
    'ECL': weighted_ecl,
    'CET1_Capital': cet1_capital,
    'Capital_Impact_Ratio': weighted_ecl / cet1_capital,
    'Capital_Buffer': cet1_capital - weighted_ecl,
    'Adequacy': 'Adequate' if weighted_ecl < cet1_capital else 'Inadequate',
    'CET1_Ratio': cet1_ratio
})

df_capital = pd.DataFrame(capital_data)
df_capital.to_csv('powerbi_exports/fact_capital_adequacy.csv', index=False)
print(f"✅ Exported fact_capital_adequacy.csv ({len(df_capital)} rows)")
audit_logger.info(f"Exported fact_capital_adequacy.csv ({len(df_capital)} rows)")



------------------------------------------------------------
3.4 FACT TABLE: CAPITAL ADEQUACY
------------------------------------------------------------
2026-09-07 18:11:09 | INFO | Exporting fact_capital_adequacy
✅ Exported fact_capital_adequacy.csv (5 rows)
2026-09-07 18:11:09 | INFO | Exported fact_capital_adequacy.csv (5 rows)


#### What It Means

This exports capital adequacy results:

- ECL under each scenario
- CET1 Capital: $45M (4.5% of $1B assets)
- Capital Impact Ratio: ECL / CET1 Capital (0.00% to 0.03%)
- Capital Buffer: CET1 - ECL
- Adequacy: Adequate or Inadequate

#### Why This Decision Was Made:
- **Why show capital adequacy by scenario?** Power BI users can see how capital holds up under stress — is there enough capital?
- **Why include the probability-weighted scenario? The probability-weighted result is the most realistic — it reflects the weighted average.
- Why include CET1 ratio (4.5%)?** This is the Basel III minimum — Power BI users need to know the baseline.
- **Why include capital buffer?** The buffer shows how much room there is before breaching the minimum.

**I export capital adequacy results so Power BI users can see the capital impact under each scenario.**

### SECTION 3.5: FACT TABLE — IFRS 9 STAGES

In [10]:
print("\n" + "-"*60)
print("3.5 FACT TABLE: IFRS 9 STAGES")
print("-"*60)
audit_logger.info("Exporting fact_ifrs9_stages")

"""
DECISION: Export IFRS 9 stage distribution with SICR details
RATIONALE: Enables regulatory reporting in Power BI
REGULATORY REFERENCE: IFRS 9 s.5.5.3, IFRS 9 s.5.5.9
"""

# Stage distribution from actual data
if 'ifrs9_stage' in df.columns:
    stage_counts = df['ifrs9_stage'].value_counts(normalize=True)
    
    stage_data = []
    for stage, pct in stage_counts.items():
        # Count by stage
        count = (df['ifrs9_stage'] == stage).sum()
        # Get ECL for this stage (if available)
        if 'estimated_ecl' in df.columns:
            avg_ecl = df[df['ifrs9_stage'] == stage]['estimated_ecl'].mean()
        else:
            avg_ecl = 0
        
        stage_data.append({
            'Stage': stage,
            'Percentage': pct,
            'Count': count,
            'Avg_ECL': avg_ecl,
            'Description': {
                'Stage 1': 'Performing (12-month ECL)',
                'Stage 2': 'SICR (Lifetime ECL)',
                'Stage 3': 'Credit Impaired (Lifetime ECL)'
            }.get(stage, 'Unknown'),
            'ECL_Horizon': {
                'Stage 1': '12-month',
                'Stage 2': 'Lifetime',
                'Stage 3': 'Lifetime'
            }.get(stage, 'Unknown'),
            'PD_Range': {
                'Stage 1': '<=0.5%',
                'Stage 2': '0.5-5%',
                'Stage 3': '>5%'
            }.get(stage, 'Unknown')
        })
else:
    # Fallback to sample data
    stage_data = [
        {'Stage': 'Stage 1', 'Percentage': 0.85, 'Count': 85000, 'Avg_ECL': 1000,
         'Description': 'Performing (12-month ECL)', 'ECL_Horizon': '12-month', 'PD_Range': '<=0.5%'},
        {'Stage': 'Stage 2', 'Percentage': 0.10, 'Count': 10000, 'Avg_ECL': 5000,
         'Description': 'SICR (Lifetime ECL)', 'ECL_Horizon': 'Lifetime', 'PD_Range': '0.5-5%'},
        {'Stage': 'Stage 3', 'Percentage': 0.05, 'Count': 5000, 'Avg_ECL': 10000,
         'Description': 'Credit Impaired (Lifetime ECL)', 'ECL_Horizon': 'Lifetime', 'PD_Range': '>5%'}
    ]

df_stages = pd.DataFrame(stage_data)
df_stages.to_csv('powerbi_exports/fact_ifrs9_stages.csv', index=False)
print(f"✅ Exported fact_ifrs9_stages.csv ({len(df_stages)} rows)")
audit_logger.info(f"Exported fact_ifrs9_stages.csv ({len(df_stages)} rows)")


------------------------------------------------------------
3.5 FACT TABLE: IFRS 9 STAGES
------------------------------------------------------------
2026-09-07 18:11:09 | INFO | Exporting fact_ifrs9_stages
✅ Exported fact_ifrs9_stages.csv (3 rows)
2026-09-07 18:11:09 | INFO | Exported fact_ifrs9_stages.csv (3 rows)


#### What It Means

This exports IFRS 9 stage distribution:

- Stage 1: Performing, 12-month ECL
- Stage 2: SICR, lifetime ECL
- Stage 3: Credit impaired, lifetime ECL

For each stage, it includes:

- Percentage of portfolio
- Count
- Average ECL
- Description
- ECL horizon (12-month vs lifetime)
- PD range

#### Why This Decision Was Made:
- **Why export stage distribution?** Power BI users can see the portfolio health — what percentage is Stage 1 vs Stage 2/3?
- **Why include ECL horizon?** This explains why ECL is different for each stage — 12-month vs lifetime.
- **Why include PD range?** This explains how loans are classified — Stage 1 is ≤0.5% PD, Stage 2 is 0.5-5%, Stage 3 is >5%.
- **Why use actual data if available?** The actual stage distribution from Notebook 03 is used if available.

**I export IFRS 9 stage distribution so Power BI users can see portfolio health and understand how loans are classified.**

### SECTION 3.6: FACT TABLE — FAIRNESS TESTING

In [11]:
print("\n" + "-"*60)
print("3.6 FACT TABLE: FAIRNESS TESTING")
print("-"*60)
audit_logger.info("Exporting fact_fairness_testing")

"""
DECISION: Export fairness testing results for Power BI
RATIONALE: Enables fair lending monitoring dashboards
REGULATORY REFERENCE: ECOA, OSFI E-23 s.6.0
"""

fairness_data = []
for attr in ['FICO_BOR', 'PROP_TYPE', 'OCCU_STAT']:
    if attr in df.columns:
        group_rates = df.groupby(attr)['default'].mean()
        min_rate = group_rates.min()
        max_rate = group_rates.max()
        disparate_impact = min_rate / max_rate if max_rate > 0 else 1.0
        
        fairness_data.append({
            'Attribute': attr,
            'Disparate_Impact_Ratio': disparate_impact,
            'Pass_80_Pct': disparate_impact >= 0.80,
            'Min_Rate': min_rate,
            'Max_Rate': max_rate,
            'Overall_Rate': df['default'].mean()
        })

df_fairness = pd.DataFrame(fairness_data)
df_fairness.to_csv('powerbi_exports/fact_fairness_testing.csv', index=False)
print(f"✅ Exported fact_fairness_testing.csv ({len(df_fairness)} rows)")
audit_logger.info(f"Exported fact_fairness_testing.csv ({len(df_fairness)} rows)")


------------------------------------------------------------
3.6 FACT TABLE: FAIRNESS TESTING
------------------------------------------------------------
2026-09-07 18:11:09 | INFO | Exporting fact_fairness_testing
✅ Exported fact_fairness_testing.csv (3 rows)
2026-09-07 18:11:09 | INFO | Exported fact_fairness_testing.csv (3 rows)


#### What It Means

This exports fairness testing results:

- FICO_BOR: Disparate Impact Ratio
- PROP_TYPE: Disparate Impact Ratio
- OCCU_STAT: Disparate Impact Ratio
- 4/5ths Rule: Pass/Fail
- Min/Max rates: The range of default rates across groups

#### Why This Decision Was Made:
- **Why export fairness testing?** ECOA requires fair lending monitoring. Power BI dashboards can track this over time.
- **Why include Pass_80_Pct?** This is the 4/5ths rule result — Power BI users can see if any attribute fails.
- **Why include min and max rates?** The range shows the disparity — how different are the best and worst groups?
- **Why include overall rate? Power BI users can compare each group to the portfolio average.

**I export fairness testing results so Power BI users can monitor disparate impact and comply with ECOA.**

### SECTION 3.7-3.8: DIMENSION TABLES & SUMMARY METRICS

In [12]:
print("\n" + "-"*60)
print("3.7 DIMENSION TABLES")
print("-"*60)
audit_logger.info("Exporting dimension tables")

"""
DECISION: Create dimension tables for Power BI relationships
RATIONALE: Enables star schema modeling in Power BI
REGULATORY REFERENCE: OSFI E-23 s.4.2
"""

# Date dimension
dates = pd.date_range(start='2020-01-01', end='2025-12-31', freq='D')
df_date = pd.DataFrame({
    'Date': dates,
    'Year': dates.year,
    'Quarter': dates.quarter,
    'Month': dates.month,
    'Month_Name': dates.strftime('%B'),
    'Day': dates.day,
    'Day_of_Week': dates.dayofweek,
    'Week': dates.isocalendar().week
})
df_date.to_csv('powerbi_exports/dim_date.csv', index=False)
print(f"✅ Exported dim_date.csv ({len(df_date)} rows)")
audit_logger.info(f"Exported dim_date.csv ({len(df_date)} rows)")

# Segment dimension
segment_dim = pd.DataFrame({
    'Segment_Type': ['FICO', 'LTV', 'Property Type', 'Loan Age', 'DTI'],
    'Segment_Description': [
        'Credit quality bands',
        'Loan-to-value ratio bands',
        'Property type categories',
        'Loan age bands',
        'Debt-to-income categories'
    ]
})
segment_dim.to_csv('powerbi_exports/dim_segment.csv', index=False)
print(f"✅ Exported dim_segment.csv ({len(segment_dim)} rows)")
audit_logger.info(f"Exported dim_segment.csv ({len(segment_dim)} rows)")


------------------------------------------------------------
3.7 DIMENSION TABLES
------------------------------------------------------------
2026-09-07 18:11:09 | INFO | Exporting dimension tables
✅ Exported dim_date.csv (2192 rows)
2026-09-07 18:11:09 | INFO | Exported dim_date.csv (2192 rows)
✅ Exported dim_segment.csv (5 rows)
2026-09-07 18:11:09 | INFO | Exported dim_segment.csv (5 rows)


#### What It Means

This exports:

- Dimension tables: Date dimension (for time-based analysis) and Segment dimension (for filtering)
- Summary metrics: 17 KPIs for executive dashboards

#### Why These Decisions Were Made:
- **Why a date dimension?** Date dimensions are standard in star schema — they enable time-based analysis (monthly trends, quarterly comparisons).
- **Why a segment dimension?** Segment dimensions are standard in star schema — they enable filtering by segment type.
- **Why 17 summary metrics?** These are the key KPIs — portfolio health, IFRS 9 parameters, capital adequacy, fairness.
- **Why include categories?** Categories help Power BI users organize the metrics — Portfolio, Risk, IFRS9, Capital, Fairness.

**I export dimension tables and summary metrics for Power BI. This enables time-based analysis and executive dashboards.**

### SECTION 3.8: SUMMARY METRICS

In [13]:
print("\n" + "-"*60)
print("3.8 SUMMARY METRICS")
print("-"*60)
audit_logger.info("Exporting summary_metrics")

"""
DECISION: Export key summary metrics for Power BI dashboard
RATIONALE: Enables executive dashboard with KPIs
REGULATORY REFERENCE: OSFI E-23 s.4.2
"""

# Calculate weighted ECL
weighted_ecl = df_scenarios['Weighted_ECL'].sum()

summary_metrics = {
    'Metric': [
        'Total Loans',
        'Default Rate',
        'Total UPB',
        'Average LTV',
        'Average DTI',
        'Average FICO',
        'PD',
        'LGD',
        'EAD',
        'ECL',
        'Probability-Weighted ECL',
        'CET1 Capital',
        'Capital Impact Ratio',
        'Stage 1 %',
        'Stage 2 %',
        'Stage 3 %',
        'Fairness PASS'
    ],
    'Value': [
        f"{len(df):,}",
        f"{df['default'].mean():.2%}",
        f"${df['ORG_UPB'].sum():,.0f}",
        f"{df['ORG_LTV'].mean():.1f}%",
        f"{df['DTI'].mean():.1f}%",
        f"{df['FICO_BOR'].mean():.0f}",
        f"{base_pd:.2%}",
        f"{base_lgd:.2%}",
        f"${base_ead:,.0f}",
        f"${base_pd * base_lgd * base_ead:,.0f}",
        f"${weighted_ecl:,.0f}",
        f"${cet1_capital:,.0f}",
        f"{weighted_ecl / cet1_capital:.2%}",
        f"{df_stages[df_stages['Stage'] == 'Stage 1']['Percentage'].values[0]*100:.1f}%" if 'Stage 1' in df_stages['Stage'].values else 'N/A',
        f"{df_stages[df_stages['Stage'] == 'Stage 2']['Percentage'].values[0]*100:.1f}%" if 'Stage 2' in df_stages['Stage'].values else 'N/A',
        f"{df_stages[df_stages['Stage'] == 'Stage 3']['Percentage'].values[0]*100:.1f}%" if 'Stage 3' in df_stages['Stage'].values else 'N/A',
        f"{len(df_fairness[df_fairness['Pass_80_Pct'] == True])}/{len(df_fairness)}"
    ],
    'Category': [
        'Portfolio', 'Portfolio', 'Portfolio',
        'Risk', 'Risk', 'Risk',
        'IFRS9', 'IFRS9', 'IFRS9',
        'IFRS9', 'IFRS9',
        'Capital', 'Capital',
        'IFRS9', 'IFRS9', 'IFRS9',
        'Fairness'
    ]
}
df_summary = pd.DataFrame(summary_metrics)
df_summary.to_csv('powerbi_exports/summary_metrics.csv', index=False)
print(f"✅ Exported summary_metrics.csv ({len(df_summary)} rows)")
audit_logger.info(f"Exported summary_metrics.csv ({len(df_summary)} rows)")



------------------------------------------------------------
3.8 SUMMARY METRICS
------------------------------------------------------------
2026-09-07 18:11:09 | INFO | Exporting summary_metrics
✅ Exported summary_metrics.csv (17 rows)
2026-09-07 18:11:09 | INFO | Exported summary_metrics.csv (17 rows)


### SECTION 4: NOTEBOOK LIMITATIONS

In [14]:
print("\n" + "="*60)
print("SECTION 4: NOTEBOOK LIMITATIONS")
print("="*60)

audit_logger.info("SECTION 4: NOTEBOOK LIMITATIONS")

print("""
===============================================================================
NOTEBOOK 06 LIMITATIONS
===============================================================================

1. STATIC EXPORT
   ---------------
   This notebook exports static CSV files for Power BI. If the underlying
   data changes, the exports must be regenerated.
   
   IMPACT: Power BI dashboards may become out of date.
   
   MITIGATION: Run this notebook whenever the source data is updated.

2. SAMPLE SIZE
   ------------
   The data exported is a 100,000-row sample. Full portfolio data would
   provide more accurate visualizations.
   
   IMPACT: Some visualizations may not capture rare events.
   
   MITIGATION: The sample size is documented and sufficient for
   demonstration purposes.

3. DATA REPRESENTATIVENESS
   -----------------------
   This project uses Fannie Mae data as a PROXY for bank portfolio data.
   See Notebook 01 — Section 2.1: US/Canada Applicability Gap for
   important differences.
   
   IMPACT: Visualizations may not be representative of all portfolios.
   
   MITIGATION: The proxy nature of the data is explicitly documented
   and caveated throughout the project.

4. ESTIMATED ECL
   ---------------
   The ECL values exported are estimates based on the methodology. Actual
   ECL for a bank portfolio would require portfolio-specific calibration.
   
   IMPACT: ECL figures are illustrative, not actual.
   
   MITIGATION: The estimated nature of the ECL is documented.

5. IFRS 9 STAGES
   --------------
   The stage distribution is based on the SICR implementation in Notebook 03.
   Different SICR triggers would produce different distributions.
   
   IMPACT: Stage distribution may differ from actual portfolio.
   
   MITIGATION: The SICR triggers are documented in the assumptions register.

===============================================================================
""")



SECTION 4: NOTEBOOK LIMITATIONS
2026-09-07 18:11:09 | INFO | SECTION 4: NOTEBOOK LIMITATIONS

NOTEBOOK 06 LIMITATIONS

1. STATIC EXPORT
   ---------------
   This notebook exports static CSV files for Power BI. If the underlying
   data changes, the exports must be regenerated.
   
   IMPACT: Power BI dashboards may become out of date.
   
   MITIGATION: Run this notebook whenever the source data is updated.

2. SAMPLE SIZE
   ------------
   The data exported is a 100,000-row sample. Full portfolio data would
   provide more accurate visualizations.
   
   IMPACT: Some visualizations may not capture rare events.
   
   MITIGATION: The sample size is documented and sufficient for
   demonstration purposes.

3. DATA REPRESENTATIVENESS
   -----------------------
   This project uses Fannie Mae data as a PROXY for bank portfolio data.
   See Notebook 01 — Section 2.1: US/Canada Applicability Gap for
   important differences.
   
   IMPACT: Visualizations may not be representative of all 

#### What It Means

This documents the limitations of the Power BI export:

- Static export — not automatically updated
- Sample size — 100,000 rows, not full portfolio
- Data representativeness — proxy data
- Estimated ECL — not actual bank ECL
- IFRS 9 stages — based on SICR triggers

#### Why This Decision Was Made:
- **Why list limitations?** Intellectual honesty — Power BI users need to understand the limitations.
- Why "static export" limitation?** The CSV files are static — if data changes, they need to be regenerated.
- **Why "sample size" limitation?** 100,000 rows is a sample — Power BI users should know this.
- **Why "estimated ECL" limitation?** ECL is estimated, not actual. Power BI users should know this.

**I document the limitations of their exports — Power BI users need to understand the context of the data.**

### SECTION 5: EXPORT COMPLETE

In [15]:
print("\n" + "="*60)
print("SECTION 5: EXPORT COMPLETE")
print("="*60)

print(f"\nRun ID: {RUN_ID}")
print(f"Exported {len(os.listdir('powerbi_exports'))} files to powerbi_exports/")
print("\nFiles ready for Power BI:")
for f in sorted(os.listdir('powerbi_exports')):
    print(f"  - {f}")

print(f"\nAudit Trail: logs/powerbi_export_audit_trail_{RUN_ID}.log")
print("\nOpen Power BI Desktop and import these CSV files")

audit_logger.info("Power BI export complete")


SECTION 5: EXPORT COMPLETE

Run ID: 20260907_181107
Exported 9 files to powerbi_exports/

Files ready for Power BI:
  - dim_date.csv
  - dim_segment.csv
  - fact_capital_adequacy.csv
  - fact_fairness_testing.csv
  - fact_ifrs9_stages.csv
  - fact_loan_portfolio.csv
  - fact_scenario_analysis.csv
  - fact_segment_analysis.csv
  - summary_metrics.csv

Audit Trail: logs/powerbi_export_audit_trail_20260907_181107.log

Open Power BI Desktop and import these CSV files
2026-09-07 18:11:09 | INFO | Power BI export complete


### SECTION 13: NOTEBOOK COMPLETE

In [16]:
print("\n" + "="*80)
print("NOTEBOOK 06 COMPLETE - POWER BI DATA EXPORT")
print("="*80)
print("""
POWER BI DATA EXPORT COMPLETE:

Sections Completed:
  ✅ Section 0: Audit Trail Setup
  ✅ Section 1: Load Data & Assumptions
  ✅ Section 2: Set Parameters
  ✅ Section 3: Export Fact Tables
     - 3.1 Loan Portfolio
     - 3.2 Segment Analysis
     - 3.3 Scenario Analysis
     - 3.4 Capital Adequacy
     - 3.5 IFRS 9 Stages
     - 3.6 Fairness Testing
  ✅ Section 4: Notebook Limitations
  ✅ Section 5: Export Complete

Files Exported:
  ✅ fact_loan_portfolio.csv
  ✅ fact_segment_analysis.csv
  ✅ fact_scenario_analysis.csv
  ✅ fact_capital_adequacy.csv
  ✅ fact_ifrs9_stages.csv
  ✅ fact_fairness_testing.csv
  ✅ dim_date.csv
  ✅ dim_segment.csv
  ✅ summary_metrics.csv

Power BI Dashboard Setup:
  1. Open Power BI Desktop
  2. Import all CSV files from powerbi_exports/
  3. Create relationships:
     - dim_date (Date) → fact_* (Date)
     - dim_segment (Segment_Type) → fact_segment_analysis (Segment_Type)
  4. Build visualizations using the fact tables

Next Step: Open Power BI Desktop and create the dashboard
""")
print("="*80)

audit_logger.info("NOTEBOOK 06 COMPLETE")


NOTEBOOK 06 COMPLETE - POWER BI DATA EXPORT

POWER BI DATA EXPORT COMPLETE:

Sections Completed:
  ✅ Section 0: Audit Trail Setup
  ✅ Section 1: Load Data & Assumptions
  ✅ Section 2: Set Parameters
  ✅ Section 3: Export Fact Tables
     - 3.1 Loan Portfolio
     - 3.2 Segment Analysis
     - 3.3 Scenario Analysis
     - 3.4 Capital Adequacy
     - 3.5 IFRS 9 Stages
     - 3.6 Fairness Testing
  ✅ Section 4: Notebook Limitations
  ✅ Section 5: Export Complete

Files Exported:
  ✅ fact_loan_portfolio.csv
  ✅ fact_segment_analysis.csv
  ✅ fact_scenario_analysis.csv
  ✅ fact_capital_adequacy.csv
  ✅ fact_ifrs9_stages.csv
  ✅ fact_fairness_testing.csv
  ✅ dim_date.csv
  ✅ dim_segment.csv
  ✅ summary_metrics.csv

Power BI Dashboard Setup:
  1. Open Power BI Desktop
  2. Import all CSV files from powerbi_exports/
  3. Create relationships:
     - dim_date (Date) → fact_* (Date)
     - dim_segment (Segment_Type) → fact_segment_analysis (Segment_Type)
  4. Build visualizations using the fact 